# Wikidata

In this notebook, we will use Wikidata to find metadata about movies. Wikidata is an open data project, like Wikipedia. While Wikipedia primarily offers text, Wikidata is focused on structured information. This can be useful for finding metadata.

## Making your first query

Let's start by looking at the Wikidata interface to see what kind of information it offers. Go to https://wikidata.org and look up the page for the The Matrix (1999). What kind of metadata does it contain?

### About Wikidata

Information on Wikidata is structured in a graph. The Matrix is a node in the graph, and so are related entities like Keanu Reeves, Lana Wachowski, Warner Bros, or the United States. Most of the statements on this page are links to other nodes in the graph. For example, it links "The Matrix" to "Keanu Reeves" through the "actor" property.  Nodes can also be abstract concepts, rather than concrete entities, such "thriller film", linked through the "genre" property.

What's typical for a linked open database like Wikidata is that properties like "actor" are themselves also nodes, with unique IDs. This means that you refer to the ID when you make a query; it solves some issues with using string labels, such as ambiguity, language variation, or meanings that depend on context.

To query Wikidata, you use a query language called SPARQL, which is designed to query knowledge graphs. SPARQL allows you to make complex queries like "rotten tomatoes review scores for movies with Keanu Reeves". Writing SPARQL queries can be difficult, but there are plenty of guides and examples online.

### Making a query

Let's use the API to query some metadata. Here is a convenience function to make a SPARQL query to Wikidata.

In [ ]:
import requests
from urllib.parse import urlencode

In [ ]:
def get_sparql(sparql: str):
    query = urlencode({'query': sparql, 'format': 'json'})
    url = f'https://query.wikidata.org/bigdata/namespace/wdq/sparql?{query}'
    return requests.get(url, headers={'User-Agent': 'INFOMPPM notebook'})

We'll try to make a query to get the actors from The Matrix. Here is a simple query to start with:

In [ ]:
query = '''
SELECT ?node
WHERE
{
  wd:Q83495 wdt:P161 ?node.
}
'''

This is hard to make sense of. You can follow the general structure of the query: it's selecting nodes, and then using `WHERE` to provide a condition of some sort. Between the brackets is a statement that must hold. It gives an item, a property and a value. Here, the item and the properties are IDs of specific nodes in the graph.

A useful tool here is https://query.wikidata.org/ . You can copy-paste the query in there. Hover over the IDs to see what they refer to. When you're writing a script to extract data, it can be useful to use the query builder first, and then scale up in a Python script.

Now, let's run our query and parse the result:

In [ ]:
res = get_sparql(query)
assert res.status_code == 200

In [ ]:
import json

In [ ]:
data = json.loads(res.content)

for result in data['results']['bindings']:
    print(result)

Each "node" in the result is a URI - you can click these links to inspect the nodes in the web interface.

If you're using actor sets to find similar movies, a list of URIs is all you need. But if we want to show actors in the interface, some names would be nice.

## Expand your query

Use the code above to make a new request. Edit the SPARQL query so it also includes the name, sex, and date of birth of each actor.

Some general tips for writing SPARQL queries:
- Take the time to look at the Wikidata web interface. In this case, look at the pages for actors. What property IDs do you need?
- You can browse many examples of SPARQL queries at https://query.wikidata.org/ - this can be useful to see how your query should be structured.

In [ ]:
# your code here

## Finding the right node

In the example above, we tried to find information related to The Matrix, by using its Wikidata ID. But if you're using Wikidata to enrich a dataset, it's likely that you won't have these IDs, so you need to search for the right nodes first.

Let's say we have a dataset from the BBC that looks like this:

| ID | Title | Year | Genre |
|----|-------|------|-------|
| 1 | Planet Earth | 2006 | documentary |
| 2 | Life | 2009 | documentary |
| 3 | Ice Age Giants | 2013 | documentary |

It would be a good idea to enrich this data, but the IDs are not going to help us. Let's make a query to find Planet Earth in Wikidata.


Below is a query to find "Planet Earth" in Wikidata by matching the title. To distinguish it from other things called "Planet Earth", we select items that were created by the BBC.

Matching titles can be tricky, so this query uses a lowercase transformation. It also checks that the title contains the search string, which is a bit more lenient than requiring a full match.

```sparql
SELECT DISTINCT ?item ?title WHERE {
  ?item wdt:P170 wd:Q9531.
  ?item wdt:P1476 ?title.
  FILTER(CONTAINS(lcase(str(?title)), "planet earth"))
}
```

Write some code to fetch the results from the query, and then adapt it to find the right Planet Earth item.

1. This query contains a small mistake, so it won't match anything. Try to identify the issue and correct the query.
2. After correcting the query, you should get two results: Planet Earth and Planet Earth II. Adapt your query so it will match the right item. You could use a stricter match for the title, or use the year in the data.

In [ ]:
# your code here

Now that we have found the right now, we can get some extra metadata. Expand your query to return some metadata about the result. This can be anything you like. You can explore [Planet Earth's Wikidata page](https://www.wikidata.org/wiki/Q377165) to see what metadata properties may be available.

In [ ]:
# your code here

Use your query to iterate over the `bbc_data` example and fetch metadata for each item. Create a new dataframe based on `bbc_data`, with extra columns based on the Wikidata results.

In [ ]:
import pandas

In [ ]:
bbc_data = pandas.DataFrame({
    'id': [1,2,3],
    'title': ['Planet Earth', 'Life', 'Ice Age Giants'],
    'year': [2006, 2009, 2013],
    'genre': ['documentary', 'documentary', 'documentary']
})
bbc_data


In [ ]:
# your code here